# Gestión de errores - Finalize Pipeline Control

## Objetivo

Este notebook tiene como objetivo registrar y actualizar el detalle de errores
generados durante la ejecución del Task `finalize_pipeline_control`.

El proceso permite mantener la trazabilidad de la ejecución en:

`prod_dataco.metadata.etl_log`

## Flujo

1. Recibe `log_id` y `run_id` de la ejecución fallida.
2. Si `log_id` no es recibido, busca el último registro en estado `EN_PROCESO`
   correspondiente a `finalize_pipeline_control`.
3. Utiliza `run_id` para consultar la API de Databricks Jobs.
4. Obtiene el detalle técnico del error generado durante la ejecución.
5. Construye el mensaje de error con la información de trazabilidad.
6. Actualiza el registro correspondiente en `etl_log`.
7. Cambia el estado de `EN_PROCESO` a `ERROR`.

## Tabla de trazabilidad

- Tabla: `prod_dataco.metadata.etl_log`
- Task origen: `finalize_pipeline_control`
- Task de error: `log_error_finalize_control`
- Estado inicial: `EN_PROCESO`
- Estado final en caso de falla: `ERROR`

## Parámetros recibidos

| Parámetro | Descripción |
|---|---|
| `log_id` | Identificador único del registro de trazabilidad |
| `run_id` | Identificador de la ejecución del Task en Databricks |

## Resultado esperado

Ante una falla de `finalize_pipeline_control`, este notebook permite
identificar la ejecución afectada y registrar en `etl_log` el mensaje técnico
devuelto por Databricks.

In [0]:
# ============================================================
# OBTENCIÓN DEL LOG_ID DEL TASK FALLIDO
# ============================================================

log_id = dbutils.jobs.taskValues.get(
    taskKey="finalize_pipeline_control",
    key="log_id",
    debugValue=""
)

print(f"log_id recibido: {log_id}")

In [0]:
# ============================================================
# RECUPERACIÓN DE LOG_ID EN CASO DE NO HABER SIDO RECIBIDO
# ============================================================

if not log_id:

    print("⚠ log_id no recibido. Buscando ejecución EN_PROCESO...")

    resultado = spark.sql("""
        SELECT log_id
        FROM prod_dataco.metadata.etl_log
        WHERE notebook = 'finalize_pipeline_control'
          AND estado = 'EN_PROCESO'
        ORDER BY fecha_inicio DESC
        LIMIT 1
    """).collect()

    if resultado:
        log_id = resultado[0]["log_id"]
        print(f"✓ log_id recuperado: {log_id}")

    else:
        raise ValueError(
            "No se encontró ningún registro EN_PROCESO "
            "para finalize_pipeline_control."
        )

In [0]:
# ============================================================
# OBTENCIÓN DEL DETALLE DEL ERROR MEDIANTE JOBS API
# ============================================================

import requests

error_detail = None

try:
    host = spark.conf.get("spark.databricks.workspaceUrl")

    token = dbutils.secrets.get(
        scope="databricks",
        key="token"
    )

    # Nota: Si necesitas la traza de un task en específico sin usar run_id,
    # se consulta el output general o se captura la traza disponible
    url = f"https://{host}/api/2.2/jobs/runs/get-output"

    headers = {
        "Authorization": f"Bearer {token}"
    }

    # Consulta el endpoint sin enviar el parámetro run_id
    response = requests.get(
        url,
        headers=headers
    )

    if response.status_code == 200:
        output_info = response.json()
        error_detail = (
            output_info.get("error")
            or output_info.get("error_trace")
            or output_info.get("notebook_output", {}).get("result")
        )
    else:
        error_detail = f"Error al consultar Jobs API ({response.status_code}): {response.text}"

except Exception as ex:
    error_detail = f"Excepción al conectar con la API: {str(ex)}"

# Traza de respaldo si no fue posible extraer el detalle
if not error_detail:
    error_detail = "No se pudo obtener el detalle del error."

print("✓ Detalle del error obtenido:")
print(error_detail)

In [0]:
# ============================================================
# CONSTRUCCIÓN DEL MENSAJE DE ERROR
# ============================================================

mensaje_error = (
    f"Task: finalize_pipeline_control | "
    f"Error code: RunExecutionError | "
    f"Resultado: failed | "
    f"Detalle: {error_detail}"
)

print("Mensaje de error:")
print(mensaje_error)

In [0]:
# ============================================================
# ACTUALIZACIÓN DEL LOG COMO ERROR
# ============================================================

from pyspark.sql import functions as F

delta_log = DeltaTable.forName(
    spark,
    "prod_dataco.metadata.etl_log"
)

(
    delta_log.alias("t")
    .update(
        condition=(
            F.col("t.log_id") == F.lit(log_id)
        ),
        set={
            "estado": "'ERROR'",
            "fecha_fin": "current_timestamp()",
            "mensaje_error": f"'{mensaje_error.replace(\"'\", \"''\")}'"
        }
    )
)

print(f"✓ etl_log actualizado como ERROR: {log_id}")